<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Lab 5 · Forecasting Connecticut's Electricity in Production
### Tune a real time-series model in Colab. Deploy it to Google Cloud. Watch it forecast the next 24 hours, every hour.

*Solo lab · about 2 hours · Colab · Bradley weather and Connecticut load since 2023, then your own energy project on Google Cloud*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/Lab5_Energy_Forecast_In_Production.ipynb)

Lab 4 priced cars. Tonight you forecast **how much electricity Connecticut will use in each of the next 24 hours**,
the same problem ISO New England (the grid operator) solves every day. You **fit a really good model** on almost four years of
history, beat three honest baselines, and **deploy it**, so it forecasts tomorrow on its own every hour from the live
weather forecast and the live load. In Lab 6 and the final project, every student's forecasts go on a class leaderboard.

| Part | Time | What you do |
|---|---|---|
| **0. Before you start** | 5 min | Your energy project is set up (pre-lab); check that all four functions deployed |
| **1. Load** | 5 min | 33,215 hours of Bradley weather + Connecticut load, Jan 2023 to Oct 15, 2026 |
| **2. Look** | 15 min | The daily, weekly and yearly rhythm, the U-shape, and why yesterday predicts today |
| **3. Features** | 10 min | `make_features`: calendar + weather + the load at the same hour yesterday (the same code runs in production) |
| **4. Baselines** | 10 min | Split by **time**, then three numbers to beat |
| **5. Tune three ways** | 25 min | Decision tree, random forest, gradient boosting, with time-series cross-validation; your champion; its loss curve |
| **6. Look at the misses** | 15 min | Real weeks, error by hour, permutation importance, the U-shape the model learned |
| **Your turn** | 10 min | Add a feature and see if it helps (you write the code) |
| **7. Ship it** | 15 min | Upload your model; run `forecast-24h` now |
| **8. Tomorrow** | 5 min | Your first real 24-hour forecast, next to ISO New England's |
| **9. Wrap up** | 5 min | Five short answers |

## The big picture
Your **energy project** (new tonight, separate from your Craigslist project) runs four Cloud Functions every hour:

| Time | Function | What it saves in `gs://YOUR-BUCKET/energy/` |
|---|---|---|
| :05 | `scraper-weather-obs` | The weather that actually happened at Bradley Airport |
| :05 | `scraper-weather-forecast` | The National Weather Service forecast for Bradley, next 24 hours |
| :15 | `scraper-isone` | Connecticut load from ISO New England (live and official), plus ISO New England's own forecast |
| **:20** | **`forecast-24h`** | **Your model's forecast for the next 24 hours** (tonight you give it a model) |

The training data you use tonight is the same kind of data, almost four years of it, cleaned step by step in
[`Lab5_Building_the_Training_Data.ipynb`](https://github.com/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/Lab5_Building_the_Training_Data.ipynb).

# Part 0 · Before you start (5 min)

**Pre-lab (before class, once, about 40 minutes):**
1. A **new Google Cloud project** for energy, with a budget alert, the same way you made your Craigslist project.
2. **Fork** [`drdave-teaching/energy-forecasting`](https://github.com/drdave-teaching/energy-forecasting) on GitHub.
3. Run the **Energy project setup guide** in Cloud Shell (the same steps as your first GCP setup, with the new names).
4. A free **ISO Express** account at [iso-ne.com](https://www.iso-ne.com/), your own (never share Dr. Dave's or a
   friend's). Then on your fork: **Settings → Secrets and variables → Actions → Secrets tab → New repository secret**,
   twice: `ISO_NE_USERNAME` and `ISO_NE_PASSWORD`. **Secrets, not Variables:** variables print in public logs.
5. **Actions** tab: run all four **Deploy** workflows once (**Run workflow → Run workflow**).

**Tonight, check (2 minutes):** your fork's **Actions** tab shows four green runs: `Deploy scraper-weather-obs`,
`Deploy scraper-weather-forecast`, `Deploy scraper-isone` and `Deploy forecast-24h`. Open the `scraper-isone` run, last
step (**Smoke invoke**): it should say `"status": "ok"`. If it says `"skipped"`, your ISO secrets are missing or misspelled:
fix them and re-run that workflow. `forecast-24h` says `"no_model"` until Part 7. That's expected.

# Part 1 · Load (5 min)

## 1a. Match production's library versions
Your model has to load inside a Cloud Function later, so we install the exact scikit-learn version `forecast-24h` uses
(pinned in [`cloud_function/forecast-24h/requirements.txt`](https://github.com/drdave-teaching/energy-forecasting/blob/main/cloud_function/forecast-24h/requirements.txt)).
If Colab asks you to **restart the session**, click it, then run this notebook again from the top.

In [ ]:
!pip install -q scikit-learn==1.6.1

import sklearn
print("scikit-learn", sklearn.__version__, "(production uses 1.6.1)")

## 1b. Your project and bucket
Type the **energy** project's ID and bucket (not your Craigslist project). Part 7 uses them to ship your model.

In [ ]:
PROJECT_ID = "YOUR-ENERGY-PROJECT-ID"     # e.g. "energy-dww05002-f26"
BUCKET_NAME = "YOUR-ENERGY-PROJECT-ID"    # usually the same as your project ID

In [ ]:
import io
import json
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module4/data/"
df = pd.read_csv(DATA + "bdl_ct_load_hourly_2023_2026.csv", index_col="hour_utc", parse_dates=["hour_utc"])
print(len(df), "hours, from", df["hour_local"].iloc[0], "to", df["hour_local"].iloc[-1])
df.head()

One row per hour, in UTC (so daylight saving never duplicates or drops an hour). `load_mw` is the average
electricity Connecticut used in that hour, in megawatts (MW). The other columns are the weather at **Bradley Airport only**, one station for the whole state.
The full data dictionary is in [`data/README.md`](https://github.com/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/data/README.md).

Want weather from more airports (other Connecticut airports, or nearby New York, Massachusetts and Rhode Island)?
That's optional, a final-project idea: the data README explains where to get them and what production would need.

# Part 2 · Look (15 min)

Time series have **rhythms**. Before fitting anything, find them. First, give every row its local hour, day and month
(people turn on lights and air conditioners on local time, not UTC).

`.dayofweek` turns a date into a number from 0 to 6, starting on **Monday**:

| Mon | Tue | Wed | Thu | Fri | Sat | Sun |
|---|---|---|---|---|---|---|
| 0 | 1 | 2 | 3 | 4 | 5 | 6 |

So `day_of_week < 5` means a weekday and `day_of_week >= 5` means the weekend (Part 3's `is_weekend` uses the same rule).

In [ ]:
local = df.index.tz_convert("America/New_York")
df["hour"] = local.hour
df["day_of_week"] = local.dayofweek        # 0 = Monday ... 6 = Sunday
df["month"] = local.month
df["year"] = local.year
print("Average load:", round(df["load_mw"].mean()), "MW | lowest hour:", round(df["load_mw"].min()),
      "MW | highest hour:", round(df["load_mw"].max()), "MW")

## 2a. The daily rhythm
Load is lowest around 3 to 4 a.m., climbs as people wake up, and peaks in the late afternoon and early evening.
Weekends start later and stay lower: offices and factories are closed.

In [ ]:
weekday = df[df["day_of_week"] < 5].groupby("hour")["load_mw"].mean()
weekend = df[df["day_of_week"] >= 5].groupby("hour")["load_mw"].mean()
plt.figure(figsize=(8, 4))
plt.plot(weekday.index, weekday.values, marker="o", label="Weekdays")
plt.plot(weekend.index, weekend.values, marker="o", label="Weekends")
plt.xticks(range(0, 24, 2))
plt.xlabel("Hour of the day (Connecticut time)")
plt.ylabel("Average load (MW)")
plt.title("The daily rhythm of Connecticut's electricity")
plt.legend()
plt.show()

## 2b. The yearly rhythm
Two peaks: **summer** (air conditioning) is the big one, **winter** (heat, lights, long nights) the smaller one.
Spring and fall are the quiet months.

In [ ]:
by_month = df.groupby("month")["load_mw"].mean()
plt.figure(figsize=(8, 4))
plt.bar(by_month.index, by_month.values, color="#1f77b4")
plt.xticks(range(1, 13), ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
plt.ylabel("Average load (MW)")
plt.title("Summer peak, smaller winter peak")
plt.show()

## 2c. The U-shape: temperature is the star weather feature
Hot hours: air conditioning. Cold hours: heating and lighting. Mild hours (55 to 65 °F): the least electricity.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df["temp_f"], df["load_mw"], s=2, alpha=0.15)
plt.xlabel("Temperature at Bradley (°F)")
plt.ylabel("Connecticut load (MW)")
plt.title("Every hour since 2023: the U-shape")
plt.show()

## 2d. Why time series are different: yesterday predicts today
In Lab 4, one car told you nothing about the next car. Here, **the past is a feature**. The load one hour ago, the load
at the same hour yesterday, and the load at the same hour last week are all strongly related to the load now. This
relationship is called **autocorrelation** (a series correlated with its own past).

In [ ]:
LAGS = {"1 hour ago": 1, "same hour yesterday (24 h)": 24, "same hour last week (168 h)": 168}
for name in LAGS:
    earlier = df["load_mw"].shift(LAGS[name])
    print("Correlation of load now with load", name + ":", round(df["load_mw"].corr(earlier), 3))

plt.figure(figsize=(5, 5))
plt.scatter(df["load_mw"].shift(24), df["load_mw"], s=2, alpha=0.1)
plt.xlabel("Load at the same hour yesterday (MW)")
plt.ylabel("Load now (MW)")
plt.title("Yesterday predicts today")
plt.show()

**Which past can we actually use?** Our forecast is made now, for each of the **next 24 hours**. The load one
hour before a target hour 20 hours away hasn't happened yet, so it's off limits. The load at the **same hour yesterday**
has (almost always) already happened: that's the one we use. Using information you wouldn't have had at forecast time
is called **leakage**, and it makes a model look great in Colab and fail in production.

# Part 3 · Features (10 min)

A feature is only useful if `forecast-24h` can build it **at forecast time**, for each of the next 24 hours.
`make_features` (below) builds the **11 features** your model uses tonight, and `forecast-24h` runs this exact function
in production (`cloud_function/forecast-24h/main.py` in your fork), so the two always agree.

### Used tonight: the 11 features

| Group | Columns | In production, it comes from |
|---|---|---|
| Calendar (Connecticut time) | `hour`, `day_of_week`, `month`, `is_weekend`, `is_holiday` | the clock |
| Weather at that hour | `temp_f`, `dewpoint_f`, `rel_humidity_pct`, `wind_speed_mph`, `cloud_cover_pct` | the National Weather Service forecast (same five, same units) |
| Load | `load_yesterday_mw` (same hour, 24 hours earlier) | your ISO-NE scraper |

### OK to add in the final project
- **More forecast weather:** `wind_gust_mph`, `wind_dir_deg`, `precip_in` (the NWS forecast has these too)
- **Features made from the allowed columns:** degree-hours `max(65 - temp_f, 0)` and `max(temp_f - 65, 0)`, `temp_f` squared,
  `temp_f` × `hour`, day of the year, `feels_like_f` computed from temperature, humidity and wind
- **Load 2 days or a week earlier** (`shift(48)`, `shift(168)`): only once your ISO-NE scraper has been running that long

### Never use
- `pressure_inhg`, `visibility_mi`, `is_rain`, `is_snow`, `is_fog`: observed only. There's no forecast, so you won't know them for tomorrow.
- **Load 1 to 23 hours earlier:** for most of the next 24 hours, that hour hasn't happened yet (leakage).
- **ISO New England's own forecast:** that's copying the professionals, not forecasting.

**Tonight, keep `FEATURES` exactly as it is.** A model that expects any other columns would stop your hourly forecasts
(Part 7a checks this for you). In the final project you can add features from the OK list: change `make_features` here
**and** in your fork's `forecast-24h/main.py` (word for word), commit, wait for **Deploy forecast-24h** to finish green,
then ship the model you trained with them. Want to try new features tonight without touching production? The
[bonus notebook](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/Lab5_Bonus_Features_Ensembles_Ranges.ipynb)
does exactly that.

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar

WEATHER = ["temp_f", "dewpoint_f", "rel_humidity_pct", "wind_speed_mph", "cloud_cover_pct"]
CALENDAR = ["hour", "day_of_week", "month", "is_weekend", "is_holiday"]
FEATURES = CALENDAR + WEATHER + ["load_yesterday_mw"]


def make_features(hours_utc, weather, load):
    """One row of model inputs per hour.
    hours_utc: the hours to describe (UTC).  weather: table indexed by UTC hour with the WEATHER columns.
    load: Connecticut load in MW, indexed by UTC hour (only hours that are already over)."""
    hours_utc = pd.DatetimeIndex(hours_utc)
    local = hours_utc.tz_convert("America/New_York")           # people use electricity on local time
    out = pd.DataFrame(index=hours_utc)
    out["hour"] = local.hour
    out["day_of_week"] = local.dayofweek                       # 0 = Monday ... 6 = Sunday
    out["month"] = local.month
    out["is_weekend"] = (local.dayofweek >= 5).astype(int)
    holidays = USFederalHolidayCalendar().holidays(start="2020-01-01", end="2030-12-31")
    out["is_holiday"] = local.tz_localize(None).normalize().isin(holidays).astype(int)

    for column in WEATHER:
        out[column] = weather[column].reindex(hours_utc).values

    # Same hour yesterday. If that hour isn't over yet (only the last hour of a live forecast),
    # use the hour just before it.
    yesterday = load.reindex(hours_utc - pd.Timedelta(hours=24)).values
    hour_before = load.reindex(hours_utc - pd.Timedelta(hours=25)).values
    out["load_yesterday_mw"] = np.where(np.isnan(yesterday), hour_before, yesterday)
    return out[FEATURES]

Build the features for all 33,215 hours. The very first day has no "yesterday", so those 24 hours are dropped.

**One honest difference:** here the weather is what **actually happened**. In production, `forecast-24h` only has the
weather **forecast**, which is sometimes wrong. So expect your live error to be a little worse than tonight's test error.

In [ ]:
X_all = make_features(df.index, df, df["load_mw"])
y_all = df["load_mw"]
has_yesterday = X_all["load_yesterday_mw"].notna()
X_all = X_all[has_yesterday]
y_all = y_all[has_yesterday]
print(len(X_all), "hours with features |", len(FEATURES), "features:", FEATURES)
X_all.head()

# Part 4 · Baselines (10 min)

## 4a. Split by time, never at random
In Lab 4 we shuffled the cars and held out 20%. **Never shuffle a time series.** Neighboring hours are almost
identical, so a shuffled test set is full of hours whose neighbors were in training: the model "remembers" instead of
forecasting. A real forecast always predicts the **future** from the **past**, so we split the same way:
- **Test:** the newest year in the data (2026 so far, Jan 1 to Oct 15), which the model never sees until the very end
- **Train:** every year before it (2023, 2024 and 2025: three full years, Connecticut time)

In [ ]:
years = X_all.index.tz_convert("America/New_York").year
TEST_YEAR = years.max()      # the newest year in the data is the test year
train = years < TEST_YEAR
test = years == TEST_YEAR
X_train = X_all[train]
X_test = X_all[test]
y_train = y_all[train]
y_test = y_all[test]
TRAIN_YEARS = str(years.min()) + "-" + str(TEST_YEAR - 1)
print(len(X_train), "training hours (" + TRAIN_YEARS + ") |", len(X_test), "test hours (" + str(TEST_YEAR) + " so far)")

## 4b. Three numbers to beat
1. **Always guess the average** (the mean training load). If a model can't beat this, it learned nothing.
2. **Same hour yesterday:** tomorrow at 5 p.m. will look like today at 5 p.m. Simple, and surprisingly hard to beat.
3. **Same hour last week:** next Tuesday at 5 p.m. will look like this Tuesday at 5 p.m.

We score with **MAE in MW** (the average miss) and **MAPE** (the average miss as a percent of the load).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error


def score(name, actual, predicted):
    return {"model": name,
            "test_mae_mw": round(mean_absolute_error(actual, predicted), 1),
            "test_mape_pct": round(100 * mean_absolute_percentage_error(actual, predicted), 2)}


mean_guess = np.full(len(y_test), y_train.mean())
same_hour_yesterday = X_test["load_yesterday_mw"]
same_hour_last_week = df["load_mw"].shift(168).reindex(X_test.index)

baselines = []
baselines.append(score("baseline: always the average", y_test, mean_guess))
baselines.append(score("baseline: same hour yesterday", y_test, same_hour_yesterday))
baselines.append(score("baseline: same hour last week", y_test, same_hour_last_week))
baseline_table = pd.DataFrame(baselines)
baseline_table

# Part 5 · Tune three ways (25 min)

## 5a. Cross-validation for time series
Lab 4's 5-fold cross-validation shuffled the cars. Here we use **`TimeSeriesSplit`**: every fold trains on the past and
validates on the stretch of time right after it, so no fold ever peeks at the future.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

SEED = 42
cv = TimeSeriesSplit(n_splits=5)
fold = 1
for fold_train, fold_valid in cv.split(X_train):
    first_valid = X_train.index[fold_valid[0]].tz_convert("America/New_York").strftime("%b %d, %Y")
    last_valid = X_train.index[fold_valid[-1]].tz_convert("America/New_York").strftime("%b %d, %Y")
    print("Fold", fold, ": train on", len(fold_train), "hours, then validate on", first_valid, "to", last_valid)
    fold = fold + 1

## 5b. Three model families, three grids
The same `GridSearchCV` as Lab 4, with `cv=TimeSeriesSplit`. All features are numbers, so no pipeline is needed.
**Gradient boosting** (`HistGradientBoostingRegressor`) is new: it builds trees **one after another**, each fixing the
previous trees' mistakes. It's the workhorse of real-world load forecasting. `max_iter` is the number of trees. We turn early stopping off
(`early_stopping=False`), so every model builds all of its trees on all the training hours; 5d shows where more trees stop helping.

This takes about 3 to 5 minutes in Colab. **Want a better model?** Add values to a grid and re-run (bigger grids take longer).
**Class challenge:** who gets the lowest `train_cv_mae_mw`?

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

candidates = {
    "decision_tree": (DecisionTreeRegressor(random_state=SEED),
                      {"max_depth": [8, 12, 16], "min_samples_leaf": [5, 20, 50]}),
    "random_forest": (RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
                      {"max_depth": [12, 20], "min_samples_leaf": [2, 10]}),
    "gradient_boosting": (HistGradientBoostingRegressor(random_state=SEED, early_stopping=False),
                          {"learning_rate": [0.05, 0.1], "max_iter": [300, 600], "max_leaf_nodes": [31, 63]}),
}

searches = {}
rows = []
for name in candidates:
    estimator = candidates[name][0]
    grid = candidates[name][1]
    started = time.time()
    search = GridSearchCV(estimator, grid, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1)
    search.fit(X_train, y_train)
    searches[name] = search
    best = search.best_estimator_
    test_pred = best.predict(X_test)
    row = score(name, y_test, test_pred)
    row["train_cv_mae_mw"] = round(-search.best_score_, 1)
    row["train_cv_std_mw"] = round(search.cv_results_["std_test_score"][search.best_index_], 1)
    row["best_params"] = str(search.best_params_)
    row["seconds"] = round(time.time() - started)
    rows.append(row)
    print(name, "done in", row["seconds"], "seconds: train_cv_mae", row["train_cv_mae_mw"], "MW")

results = pd.DataFrame(rows).sort_values("train_cv_mae_mw")
results[["model", "train_cv_mae_mw", "train_cv_std_mw", "test_mae_mw", "test_mape_pct", "best_params", "seconds"]]

## 5c. Your champion
The model with the lowest `train_cv_mae_mw` wins. The test period (2026 so far) is your final exam: you don't choose with it.
Want to ship a different one? Uncomment the second line.

In [ ]:
CHAMPION = results.iloc[0]["model"]
# CHAMPION = "random_forest"       # <-- uncomment to override: "decision_tree", "random_forest" or "gradient_boosting"

champion = searches[CHAMPION].best_estimator_
champion_pred = champion.predict(X_test)
champion_score = score("champion: " + CHAMPION, y_test, champion_pred)
champion_train_cv_mae = -searches[CHAMPION].best_score_
print("Champion:", CHAMPION, searches[CHAMPION].best_params_)

scoreboard = pd.concat([baseline_table, pd.DataFrame([champion_score])], ignore_index=True)
scoreboard = scoreboard.sort_values("test_mae_mw")
scoreboard

## 5d. The loss curve: watch boosting learn
Gradient boosting adds trees one at a time, so we can **watch it learn**. We train on the earlier 80% of the training
hours and track the error, tree by tree, on the training hours and on the **later** 20% (validation). The training error
keeps falling forever; the validation error falls, then flattens. Where it flattens, more trees stop helping.
(If the lowest point is the very last tree, more trees might still help a little: try a bigger `max_iter` in the grid.)

In [ ]:
boost_params = searches["gradient_boosting"].best_params_
cut = int(len(X_train) * 0.8)                      # earlier 80% = fit, later 20% = validation (never shuffled)
X_fit = X_train.iloc[:cut]
y_fit = y_train.iloc[:cut]
X_valid = X_train.iloc[cut:]
y_valid = y_train.iloc[cut:]

booster = HistGradientBoostingRegressor(random_state=SEED, early_stopping=False, **boost_params)
history = booster.fit(X_fit, y_fit)                # fit() returns the fitted model, kept as "history"

fit_mae = []
valid_mae = []
for fit_pred in history.staged_predict(X_fit):     # the prediction after 1 tree, 2 trees, 3 trees, ...
    fit_mae.append(mean_absolute_error(y_fit, fit_pred))
for valid_pred in history.staged_predict(X_valid):
    valid_mae.append(mean_absolute_error(y_valid, valid_pred))

trees = np.arange(1, len(fit_mae) + 1)
best_trees = int(np.argmin(valid_mae)) + 1
plt.figure(figsize=(8, 4))
plt.plot(trees, fit_mae, label="training hours")
plt.plot(trees, valid_mae, label="validation hours (later in time)")
plt.axvline(best_trees, color="gray", linestyle="--", label="lowest validation error: " + str(best_trees) + " trees")
plt.ylim(0, max(valid_mae[10], fit_mae[10]) * 1.2)
plt.xlabel("Number of trees")
plt.ylabel("MAE (MW)")
plt.title("Loss curve: gradient boosting, tree by tree")
plt.legend()
plt.show()
print("Validation MAE: after 10 trees", round(valid_mae[9]), "MW | best", round(min(valid_mae)), "MW at", best_trees, "trees")

# Part 6 · Look at the misses (15 min)

## 6a. The hottest and the coldest week of the test year
The code finds the hottest and the coldest hour of the test year and shows the week around each: the actual load, your
champion, and "same hour yesterday".

In [ ]:
test_results = pd.DataFrame({"actual": y_test, "champion": champion_pred,
                             "same_hour_yesterday": X_test["load_yesterday_mw"], "temp_f": X_test["temp_f"]},
                            index=X_test.index)
test_results.index = test_results.index.tz_convert("America/New_York")

hottest = test_results["temp_f"].idxmax()
coldest = test_results["temp_f"].idxmin()
WEEKS = {"Hottest week of " + str(TEST_YEAR): hottest, "Coldest week of " + str(TEST_YEAR): coldest}
for title in WEEKS:
    center = WEEKS[title]
    week = test_results.loc[center - pd.Timedelta(days=3):center + pd.Timedelta(days=3)]
    label = center.strftime("%b %d, %I %p") + ", " + str(round(test_results.loc[center, "temp_f"])) + " °F"
    plt.figure(figsize=(10, 3.5))
    plt.plot(week.index, week["actual"], color="black", label="actual")
    plt.plot(week.index, week["champion"], label="champion (" + CHAMPION + ")")
    plt.plot(week.index, week["same_hour_yesterday"], alpha=0.6, linestyle=":", label="same hour yesterday")
    plt.ylabel("Load (MW)")
    plt.title(title + " (centered on " + label + ")")
    plt.legend()
    plt.show()

## 6b. When does it miss most?
Each hour's miss as a **percent of the load** that hour, so a 100 MW miss at 3 a.m. (when Connecticut uses less)
counts more than the same miss at 5 p.m.

In [ ]:
test_results["abs_error"] = (test_results["champion"] - test_results["actual"]).abs()
test_results["pct_error"] = 100 * test_results["abs_error"] / test_results["actual"]
by_hour = test_results.groupby(test_results.index.hour)["pct_error"].mean()
plt.figure(figsize=(8, 3.5))
plt.bar(by_hour.index, by_hour.values)
plt.xticks(range(0, 24, 2))
plt.xlabel("Hour of the day (Connecticut time)")
plt.ylabel("Average miss (% of the load)")
plt.title("Champion's error by hour of the day (" + str(TEST_YEAR) + ")")
plt.show()

print("The 5 worst hours of", TEST_YEAR, "so far:")
test_results.sort_values("pct_error", ascending=False).head(5).round(1)

## 6c. Which features matter? Permutation importance
Shuffle one feature's column in the test set and see how much worse the model gets. A big jump means the model leans on
that feature. Each box shows 10 shuffles.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(champion, X_test, y_test, n_repeats=10, random_state=SEED,
                              scoring="neg_mean_absolute_error", n_jobs=-1)
order = perm.importances_mean.argsort()
box_data = []
box_labels = []
for i in order:
    box_data.append(perm.importances[i])
    box_labels.append(FEATURES[i])
plt.figure(figsize=(8, 5))
plt.boxplot(box_data, vert=False)
plt.yticks(range(1, len(box_labels) + 1), box_labels)
plt.xlabel("How much worse the MAE gets when shuffled (MW)")
plt.title("Permutation importance (" + CHAMPION + ", test year)")
plt.show()

## 6d. Did the model learn the U-shape?
A partial dependence plot holds everything else fixed and slides one feature across its range. If the model learned
the physics, temperature should come out as a U, and the hour of the day as the daily rhythm from Part 2.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
PartialDependenceDisplay.from_estimator(champion, X_train.sample(3000, random_state=SEED), ["temp_f", "hour"], ax=ax)
fig.suptitle("What " + CHAMPION + " learned: temperature and hour of the day")
plt.show()

## Your turn: does last week help? (10 min, you write the code)
Part 2d showed the load at the **same hour last week** is also strongly related to the load now. Add it as a feature
and find out if your champion gets better. Write the code yourself (ask Claude if you're stuck, but type it in):
1. Make the new column: `df["load_mw"].shift(168)`, lined up with `X_train` and `X_test` (hint: `.reindex(X_train.index)`).
2. Copy `X_train` and `X_test`, add the column to each copy, and call it `load_last_week_mw`.
3. Fit a fresh model with your champion's best settings (`type(champion)(**champion.get_params())` makes an unfitted copy),
   then compare its test MAE with the champion's.

**Before you celebrate:** `forecast-24h` can only use last week's load after your ISO-NE scraper has been running for a
full week. Tonight's production model keeps the original features; you can add this one in the final project.

In [ ]:
# YOUR CODE HERE: add load_last_week_mw, refit, and print both test MAEs

# Part 7 · Ship it (15 min)

## 7a. Save the model and its model card
`model.joblib` is your champion, frozen. `model_card.json` is its label: what it is, how good it was, and which features
and library version it needs.

In [ ]:
from datetime import datetime, timezone

# forecast-24h builds exactly these 11 columns (cloud_function/forecast-24h/main.py in your fork)
PRODUCTION_FEATURES = ["hour", "day_of_week", "month", "is_weekend", "is_holiday", "temp_f", "dewpoint_f",
                       "rel_humidity_pct", "wind_speed_mph", "cloud_cover_pct", "load_yesterday_mw"]
if list(champion.feature_names_in_) != PRODUCTION_FEATURES:
    raise ValueError("Your champion was trained on " + str(list(champion.feature_names_in_)) + ", but forecast-24h "
                     "builds " + str(PRODUCTION_FEATURES) + ". Put FEATURES in Part 3 back and re-run, or change "
                     "make_features in your fork first (see the rules in Part 3).")

joblib.dump(champion, "/content/model.joblib", compress=3)
best_params = {}
for key in searches[CHAMPION].best_params_:
    best_params[key] = str(searches[CHAMPION].best_params_[key])
card = {
    "model_type": type(champion).__name__,
    "champion": CHAMPION,
    "best_params": best_params,
    "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sklearn_version": sklearn.__version__,
    "features": FEATURES,
    "train_cv_mae_mw": round(float(champion_train_cv_mae), 1),
    "test_mae_mw": champion_score["test_mae_mw"],
    "test_mape_pct": champion_score["test_mape_pct"],
    "baseline_same_hour_yesterday_mae_mw": float(baseline_table.loc[1, "test_mae_mw"]),
    "train_period": TRAIN_YEARS + " (Connecticut time)",
    "test_period": test_results.index[0].strftime("%Y-%m-%d") + " to " + test_results.index[-1].strftime("%Y-%m-%d"),
    "seed": SEED,
}
with open("/content/model_card.json", "w") as f:
    json.dump(card, f, indent=2)
print("Saved /content/model.joblib (" + str(round(len(open("/content/model.joblib", "rb").read()) / 1e6, 1)) + " MB) and /content/model_card.json")
card

## 7b. Sign in and check (3 checks)
When the **"Allow access"** pop-up opens, pick the **Gmail you used for Google Cloud**. The cell checks **who** you are,
that you can see **your energy bucket**, and that **`forecast-24h`** is deployed.
No energy project yet? Set `UPLOAD = False`: Part 8 runs in demo mode.

In [ ]:
UPLOAD = True

if UPLOAD and ("YOUR-ENERGY-PROJECT-ID" in PROJECT_ID or "YOUR-ENERGY-PROJECT-ID" in BUCKET_NAME):
    print("Type your PROJECT_ID and BUCKET_NAME in Part 1b, re-run that cell, then re-run this one.")
    print("Staying in demo mode until you do.")
    UPLOAD = False

if UPLOAD:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()

    # CHECK 1: which Google account did you just pick in the pop-up?
    account_lines = !gcloud config get-value account
    signed_in_as = "unknown"
    if len(account_lines) > 0:
        signed_in_as = account_lines[-1].strip()
    print("1. Signed in as:", signed_in_as)

    # CHECK 2: can that account see your bucket?
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)
    try:
        bucket_ok = bucket.exists()
    except Exception as err:
        bucket_ok = False
        print("   Google says:", str(err)[:200])
    if not bucket_ok:
        raise RuntimeError("The account " + signed_in_as + " cannot see the bucket " + BUCKET_NAME + ". "
                           "Check BUCKET_NAME in Part 1b, or Runtime > Disconnect and delete runtime, then run again "
                           "and pick your Google Cloud Gmail in the pop-up.")
    print("2. Bucket found:", BUCKET_NAME)

    # CHECK 3: is forecast-24h deployed?
    state_lines = !gcloud functions describe forecast-24h --gen2 --region=us-central1 --project={PROJECT_ID} --format="value(state)"
    state = "NOT FOUND"
    if len(state_lines) > 0 and state_lines[-1].strip() in ["ACTIVE", "DEPLOYING", "FAILED"]:
        state = state_lines[-1].strip()
    print("3. forecast-24h:", state)
    if state != "ACTIVE":
        print("   Go to Part 0: run 'Deploy forecast-24h' on your fork's Actions tab, wait for green, then re-run this cell.")

## 7c. Upload your model
`energy/models/model.joblib` is the **live** model (replaced every time you ship); `energy/models/history/` keeps a copy
of every model you ever shipped, named by time.

In [ ]:
if UPLOAD:
    bucket.blob("energy/models/model.joblib").upload_from_filename("/content/model.joblib")
    bucket.blob("energy/models/model_card.json").upload_from_filename("/content/model_card.json")
    stamp = card["trained_at"].replace(":", "-")
    bucket.blob("energy/models/history/model_" + stamp + ".joblib").upload_from_filename("/content/model.joblib")
    bucket.blob("energy/models/history/model_card_" + stamp + ".json").upload_from_filename("/content/model_card.json")
    print("Live model:  gs://" + BUCKET_NAME + "/energy/models/model.joblib")
    print("Kept a copy: gs://" + BUCKET_NAME + "/energy/models/history/model_" + stamp + ".joblib")
else:
    print("UPLOAD is False: nothing uploaded. Part 8 will run in demo mode.")

## 7d. Run `forecast-24h` now
Normally it runs by itself every hour at :20. This cell starts it **right now**, then checks your bucket every 15 seconds.

**What `forecast-24h` does each time it runs** (`cloud_function/forecast-24h/main.py` in your fork):
1. Loads your live model from `energy/models/model.joblib`.
2. Picks the **next 24 full hours** (at 2:20 p.m. it forecasts 3 p.m. today through 2 p.m. tomorrow).
3. Builds the features with **the same `make_features`** as Part 3: the newest weather **forecast** for each hour, and the
   load at the same hour yesterday from your ISO-NE scraper (live five-minute load, averaged to the hour).
4. Writes 24 rows to `energy/forecasts/latest.csv`, a copy named by time in `energy/forecasts/history/`, and appends to
   `energy/datasets/forecasts_master.csv`. Lab 6 reads `latest.csv` for the class leaderboard.

In [ ]:
from zoneinfo import ZoneInfo

if UPLOAD:
    started = datetime.now(timezone.utc)
    !gcloud scheduler jobs run forecast-24h-hourly --location=us-central1 --project={PROJECT_ID}
    print("Started forecast-24h. Waiting for your forecast", end="")
    latest_blob = bucket.blob("energy/forecasts/latest.csv")
    ready = False
    for attempt in range(16):                 # every 15 seconds, for up to 4 minutes
        time.sleep(15)
        print(".", end="")
        if latest_blob.exists():
            latest_blob.reload()
            if latest_blob.updated > started:
                ready = True
                break
    print()
    if ready:
        print("Done at", latest_blob.updated.astimezone(ZoneInfo("America/New_York")).strftime("%I:%M %p"), "(Eastern): gs://" + BUCKET_NAME + "/energy/forecasts/latest.csv")
    else:
        print("No new forecast after 4 minutes. Run the log cell below and paste the log into Claude.")
else:
    print("UPLOAD is False: skipping. Part 8 will run in demo mode.")

**Optional, only for troubleshooting.** `"no_load_data"` means your ISO-NE secrets are missing (Part 0);
`"no_model"` means 7c didn't run; anything with `Error` is worth pasting into Claude.

In [ ]:
if UPLOAD:
    !gcloud functions logs read forecast-24h --region=us-central1 --gen2 --limit=20 --project={PROJECT_ID}
else:
    print("UPLOAD is False: no log to read.")

## 7e. Join the class leaderboard (1 cell)
Dr. Dave's collector copies every student's forecasts into one class table every hour and scores them on the
**class leaderboard** (from tonight), then Lab 6 and the final project. It needs **read-only** access to your bucket: it
can read your files, never change or delete them, and it can't touch anything else in your project.

Type your NetID and a **nickname** (what the public leaderboard shows instead of your name: keep it class-friendly,
no commas), run the cell, and paste the line it prints into the **Submit Lab 5** text box.

In [ ]:
NETID = "abc12345"            # <-- your UConn NetID, lower case
NICKNAME = "Watt Wizard"      # <-- what the leaderboard shows (up to 24 characters)
COLLECTOR = "collector@opim5512-collector-f26.iam.gserviceaccount.com"

if UPLOAD:
    !gcloud storage buckets add-iam-policy-binding gs://{BUCKET_NAME} --member=serviceAccount:{COLLECTOR} --role=roles/storage.objectViewer --project={PROJECT_ID} --format=none
    print("Shared gs://" + BUCKET_NAME + " (read-only) with Dr. Dave's collector.")
    print()
    print("Paste this line into the Submit Lab 5 text box:")
    print(NETID.strip().lower() + "," + BUCKET_NAME + "," + NICKNAME.replace(",", " ").strip()[:24])
    print()
    print("Class leaderboard (your first scored hour shows up after the next :35 run with your line on the roster):")
    print("https://storage.googleapis.com/opim5512-leaderboard-f26/index.html")
else:
    print("UPLOAD is False: nothing shared.")

# Part 8 · Tomorrow (5 min)
Your model's forecast for the next 24 hours, with the last two days of actual load before it, and **ISO New England's
own forecast** for the same hours (from your ISO-NE scraper). No upload? Demo mode forecasts the last day of the test period (Oct 15, 2026) instead.

In [ ]:
# PLUMBING (run me): load your forecast and the live data, or build a demo from the test year


def read_bucket_csv(key):
    blob = bucket.blob(key)
    if not blob.exists():
        return None
    return pd.read_csv(io.BytesIO(blob.download_as_bytes()))


LIVE = False
if UPLOAD and latest_blob.exists():
    mine = read_bucket_csv("energy/forecasts/latest.csv")
    mine["target_hour_utc"] = pd.to_datetime(mine["target_hour_utc"], utc=True)
    forecast = mine.set_index("target_hour_utc")["predicted_load_mw"]
    actual_table = read_bucket_csv("energy/datasets/ct_load_5min_hourly_master.csv")
    actual_table["hour_utc"] = pd.to_datetime(actual_table["hour_utc"], utc=True)
    actual = actual_table.set_index("hour_utc")["load_mw_5min_avg"].sort_index().tail(48)
    isone_table = read_bucket_csv("energy/datasets/isone_ct_forecast_master.csv")
    isone = None
    if isone_table is not None:
        isone_table["target_hour_utc"] = pd.to_datetime(isone_table["target_hour_utc"], utc=True)
        isone_table = isone_table.sort_values(["target_hour_utc", "issued_at_utc"])
        isone = isone_table.drop_duplicates("target_hour_utc", keep="last").set_index("target_hour_utc")["isone_forecast_mw"]
        isone = isone.reindex(forecast.index)
    LIVE = True
    print("LIVE: your model's forecast, issued", mine["issued_at_utc"].iloc[0], "(UTC)")
else:
    last_day = X_test.index[-24:]
    forecast = pd.Series(champion.predict(X_test.loc[last_day]), index=last_day)
    actual = y_test.loc[X_test.index[-72:-24]]
    isone = None
    print("DEMO: forecasting Oct 15, 2026 from the test period (upload a model in Part 7 for the real thing)")

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(actual.index.tz_convert("America/New_York"), actual.values, color="black", label="actual load (last 48 hours)")
plt.plot(forecast.index.tz_convert("America/New_York"), forecast.values, marker="o", label="YOUR forecast (next 24 hours)")
if isone is not None and isone.notna().any():
    plt.plot(isone.index.tz_convert("America/New_York"), isone.values, linestyle="--", label="ISO New England's forecast")
plt.ylabel("Connecticut load (MW)")
plt.title("Your 24-hour forecast for Connecticut")
plt.legend()
plt.show()
peak_hour = forecast.idxmax().tz_convert("America/New_York").strftime("%a %I %p")
print("Your forecast peaks at", round(forecast.max()), "MW on", peak_hour, "(Connecticut time)")

# Part 9 · Wrap up (5 min)

## Five short answers (1 or 2 sentences each)
**Your Lab 5 submission:** in the **Submit Lab 5** text box, paste your Part 7e line, then these five answers
(numbered 1 to 5). No file upload.

1. Which baseline was hardest to beat, and by how many MW did your champion beat it on the test year?
2. Look at your loss curve: about how many trees did gradient boosting need before more trees stopped helping?
3. Which feature mattered most in the permutation importance? Does that match what you saw in Part 2?
4. Your turn: did last week's load help? By how much? Why can't production use it yet?
5. Part 8: when does your forecast say tomorrow's peak will be? Is ISO New England's forecast higher or lower than yours?

*Your answers:*

---

## What happens next
Your model now forecasts the next 24 hours **every hour**, on its own, and from tonight (Part 7e) the class leaderboard
scores every forecast hour against what Connecticut actually used, next to two baselines: "same hour yesterday" and
ISO New England's own forecast. Far-ahead hours count as much as next-hour ones. In **Lab 6** you'll build a stronger
model and give a **low, expected and high** value for every hour; the final project is a week-long competition scored
on both the expected value and how honest and narrow your low-high range is.

**Data credits:** weather from the Iowa State Mesonet (mesonet.agron.iastate.edu) and the National Weather Service
(forecast.weather.gov); load and forecasts from ISO New England (iso-ne.com, ISO Express web services).